# MiniBank Integration Testing Lab
**One-day workshop | Python + Flask + SQLite + pytest + Apache JMeter**

This notebook demonstrates a small end-to-end banking transfer flow:
1. Run a simulated banking API.
2. Validate functional and integration behavior with `pytest`.
3. Execute concurrent HTTP transactions with Apache JMeter.
4. Review response time, throughput, and error rate.

**No local installation is required.** Run this notebook in Google Colab. The runtime installs Java and JMeter temporarily.

> Educational simulation only. Do not use real banking data or treat the results as production load-test evidence.

## 1. Set up the environment

This installs Flask, pytest, Java, and the JMeter command-line distribution in the temporary Colab runtime.

In [ ]:
!pip -q install flask pytest requests pandas

import os, subprocess, time, shutil, pathlib, textwrap, json
from pathlib import Path

# Install Java and download Apache JMeter
!apt-get -qq update
!apt-get -qq install -y openjdk-17-jre-headless
!wget -q https://archive.apache.org/dist/jmeter/binaries/apache-jmeter-5.6.3.tgz -O /content/jmeter.tgz
!tar -xzf /content/jmeter.tgz -C /content

JMETER = "/content/apache-jmeter-5.6.3/bin/jmeter"
print(subprocess.check_output([JMETER, "--version"], text=True))

## 2. Create and start the simulated banking API

The API exposes:
- `GET /health`
- `GET /accounts/<account_id>`
- `POST /transfer`

A transfer debits one account and credits another in a single SQLite transaction.

In [ ]:
%%writefile /content/app.py
from flask import Flask, request, jsonify
import sqlite3

app = Flask(__name__)
DB = "/content/minibank.db"

def connect():
    con = sqlite3.connect(DB, timeout=10)
    con.row_factory = sqlite3.Row
    return con

def initialize():
    with connect() as con:
        con.execute("""CREATE TABLE IF NOT EXISTS accounts (
            account_id TEXT PRIMARY KEY, balance INTEGER NOT NULL)""")
        con.execute("""CREATE TABLE IF NOT EXISTS transactions (
            id INTEGER PRIMARY KEY AUTOINCREMENT,
            source TEXT, destination TEXT, amount INTEGER, status TEXT)""")
        con.execute("INSERT OR IGNORE INTO accounts VALUES ('A001', 10000000)")
        con.execute("INSERT OR IGNORE INTO accounts VALUES ('B001', 5000000)")

@app.get("/health")
def health():
    return jsonify(status="UP")

@app.get("/accounts/<account_id>")
def account(account_id):
    with connect() as con:
        row = con.execute("SELECT * FROM accounts WHERE account_id=?",
                          (account_id,)).fetchone()
    if row is None:
        return jsonify(error="Account not found"), 404
    return jsonify(account_id=row["account_id"], balance=row["balance"])

@app.post("/transfer")
def transfer():
    data = request.get_json(silent=True) or {}
    source, destination = data.get("source"), data.get("destination")
    amount = data.get("amount")
    if not source or not destination or not isinstance(amount, int) or amount <= 0:
        return jsonify(error="Invalid transfer request"), 400
    if source == destination:
        return jsonify(error="Source and destination must differ"), 400

    con = connect()
    try:
        con.execute("BEGIN IMMEDIATE")
        src = con.execute("SELECT balance FROM accounts WHERE account_id=?",
                          (source,)).fetchone()
        dst = con.execute("SELECT balance FROM accounts WHERE account_id=?",
                          (destination,)).fetchone()
        if src is None or dst is None:
            con.rollback()
            return jsonify(error="Account not found"), 404
        if src["balance"] < amount:
            con.rollback()
            return jsonify(error="Insufficient balance"), 409

        con.execute("UPDATE accounts SET balance=balance-? WHERE account_id=?",
                    (amount, source))
        con.execute("UPDATE accounts SET balance=balance+? WHERE account_id=?",
                    (amount, destination))
        cur = con.execute(
            "INSERT INTO transactions(source,destination,amount,status) VALUES(?,?,?,?)",
            (source, destination, amount, "SUCCESS"))
        con.commit()
        return jsonify(transaction_id=cur.lastrowid, status="SUCCESS",
                       amount=amount), 200
    except Exception:
        con.rollback()
        return jsonify(error="Transaction processing failed"), 500
    finally:
        con.close()

initialize()

if __name__ == "__main__":
    app.run(host="127.0.0.1", port=5000, threaded=True)

In [ ]:
# Start the API in the background and wait until it is ready
import requests

log = open("/content/flask.log", "w")
server = subprocess.Popen(
    ["python", "/content/app.py"],
    stdout=log, stderr=subprocess.STDOUT
)

for attempt in range(30):
    try:
        response = requests.get("http://127.0.0.1:5000/health", timeout=1)
        if response.ok:
            break
    except requests.RequestException:
        time.sleep(1)
else:
    raise RuntimeError("Banking API did not start. Check /content/flask.log")

print("Banking API:", response.json())

## 3. Run functional and integration tests with pytest

These tests check a successful transfer, insufficient balance, invalid accounts, and the resulting account balances.

In [ ]:
%%writefile /content/test_banking.py
import requests

BASE = "http://127.0.0.1:5000"

def test_health_check():
    assert requests.get(f"{BASE}/health").json()["status"] == "UP"

def test_successful_transfer():
    before_a = requests.get(f"{BASE}/accounts/A001").json()["balance"]
    before_b = requests.get(f"{BASE}/accounts/B001").json()["balance"]
    result = requests.post(f"{BASE}/transfer", json={
        "source": "A001", "destination": "B001", "amount": 1000
    })
    assert result.status_code == 200
    assert result.json()["status"] == "SUCCESS"
    after_a = requests.get(f"{BASE}/accounts/A001").json()["balance"]
    after_b = requests.get(f"{BASE}/accounts/B001").json()["balance"]
    assert after_a == before_a - 1000
    assert after_b == before_b + 1000

def test_insufficient_balance_is_rejected():
    result = requests.post(f"{BASE}/transfer", json={
        "source": "A001", "destination": "B001", "amount": 999999999
    })
    assert result.status_code == 409

def test_unknown_account_is_rejected():
    result = requests.post(f"{BASE}/transfer", json={
        "source": "UNKNOWN", "destination": "B001", "amount": 100
    })
    assert result.status_code == 404

In [ ]:
!pytest -q /content/test_banking.py

## 4. Generate and execute the JMeter performance test

The test plan simulates 10 virtual users, each making 5 transfer requests. This is a small demonstration load, not a capacity benchmark.

In [ ]:
jmx = r'''<?xml version="1.0" encoding="UTF-8"?>
<jmeterTestPlan version="1.2" properties="5.0" jmeter="5.6.3">
  <hashTree>
    <TestPlan guiclass="TestPlanGui" testclass="TestPlan" testname="MiniBank Test Plan" enabled="true">
      <stringProp name="TestPlan.comments">MiniBank HTTP transfer load test</stringProp>
      <boolProp name="TestPlan.functional_mode">false</boolProp>
      <boolProp name="TestPlan.tearDown_on_shutdown">true</boolProp>
      <boolProp name="TestPlan.serialize_threadgroups">false</boolProp>
      <elementProp name="TestPlan.user_defined_variables" elementType="Arguments">
        <collectionProp name="Arguments.arguments"/>
      </elementProp>
    </TestPlan>
    <hashTree>
      <ThreadGroup guiclass="ThreadGroupGui" testclass="ThreadGroup" testname="Virtual Users" enabled="true">
        <stringProp name="ThreadGroup.num_threads">10</stringProp>
        <stringProp name="ThreadGroup.ramp_time">5</stringProp>
        <boolProp name="ThreadGroup.same_user_on_next_iteration">true</boolProp>
        <stringProp name="ThreadGroup.on_sample_error">continue</stringProp>
        <elementProp name="ThreadGroup.main_controller" elementType="LoopController">
          <boolProp name="LoopController.continue_forever">false</boolProp>
          <stringProp name="LoopController.loops">5</stringProp>
        </elementProp>
      </ThreadGroup>
      <hashTree>
        <HTTPSamplerProxy guiclass="HttpTestSampleGui" testclass="HTTPSamplerProxy" testname="POST Transfer" enabled="true">
          <boolProp name="HTTPSampler.postBodyRaw">true</boolProp>
          <elementProp name="HTTPsampler.Arguments" elementType="Arguments">
            <collectionProp name="Arguments.arguments">
              <elementProp name="" elementType="HTTPArgument">
                <boolProp name="HTTPArgument.always_encode">false</boolProp>
                <stringProp name="Argument.value">{"source":"A001","destination":"B001","amount":1}</stringProp>
                <stringProp name="Argument.metadata">=</stringProp>
              </elementProp>
            </collectionProp>
          </elementProp>
          <stringProp name="HTTPSampler.domain">127.0.0.1</stringProp>
          <stringProp name="HTTPSampler.port">5000</stringProp>
          <stringProp name="HTTPSampler.protocol">http</stringProp>
          <stringProp name="HTTPSampler.path">/transfer</stringProp>
          <stringProp name="HTTPSampler.method">POST</stringProp>
          <boolProp name="HTTPSampler.follow_redirects">true</boolProp>
          <boolProp name="HTTPSampler.use_keepalive">true</boolProp>
        </HTTPSamplerProxy>
        <hashTree>
          <HeaderManager guiclass="HeaderPanel" testclass="HeaderManager" testname="JSON Header" enabled="true">
            <collectionProp name="HeaderManager.headers">
              <elementProp name="Content-Type" elementType="Header">
                <stringProp name="Header.name">Content-Type</stringProp>
                <stringProp name="Header.value">application/json</stringProp>
              </elementProp>
            </collectionProp>
          </HeaderManager>
          <hashTree/>
        </hashTree>
      </hashTree>
      <ResultCollector guiclass="SimpleDataWriter" testclass="ResultCollector" testname="JTL Results" enabled="true">
        <boolProp name="ResultCollector.error_logging">false</boolProp>
        <objProp>
          <name>saveConfig</name>
          <value class="SampleSaveConfiguration">
            <time>true</time><latency>true</latency><timestamp>true</timestamp>
            <success>true</success><label>true</label><code>true</code>
            <message>true</message><threadName>true</threadName>
            <dataType>true</dataType><encoding>false</encoding>
            <assertions>true</assertions><subresults>true</subresults>
            <responseData>false</responseData><samplerData>false</samplerData>
            <xml>false</xml><fieldNames>true</fieldNames>
            <responseHeaders>false</responseHeaders><requestHeaders>false</requestHeaders>
            <responseDataOnError>false</responseDataOnError>
            <saveAssertionResultsFailureMessage>true</saveAssertionResultsFailureMessage>
            <bytes>true</bytes><sentBytes>true</sentBytes>
            <url>true</url><threadCounts>true</threadCounts>
            <idleTime>true</idleTime><connectTime>true</connectTime>
          </value>
        </objProp>
        <stringProp name="filename">/content/results.jtl</stringProp>
      </ResultCollector>
      <hashTree/>
    </hashTree>
  </hashTree>
</jmeterTestPlan>'''
Path("/content/banking-test.jmx").write_text(jmx)

# Remove outputs from any previous run
for item in ["/content/results.jtl", "/content/jmeter-report"]:
    p = Path(item)
    if p.is_dir():
        shutil.rmtree(p)
    elif p.exists():
        p.unlink()

run = subprocess.run([
    JMETER, "-n",
    "-t", "/content/banking-test.jmx",
    "-l", "/content/results.jtl",
    "-e", "-o", "/content/jmeter-report"
], capture_output=True, text=True)

print(run.stdout[-4000:])
if run.returncode != 0:
    print(run.stderr[-4000:])
    raise RuntimeError("JMeter execution failed")
print("JMeter test completed.")

## 5. Review the performance results

The summary reports sample count, success rate, average and 95th-percentile response time, throughput, and errors.

In [ ]:
import pandas as pd
from IPython.display import display, HTML

df = pd.read_csv("/content/results.jtl")
df["success"] = df["success"].astype(str).str.lower().eq("true")
elapsed = df["timeStamp"].max() - df["timeStamp"].min()
duration_seconds = max((elapsed + df["elapsed"].max()) / 1000, 0.001)

summary = pd.DataFrame([{
    "Samples": len(df),
    "Successful": int(df["success"].sum()),
    "Failed": int((~df["success"]).sum()),
    "Success rate (%)": round(df["success"].mean() * 100, 2),
    "Average response (ms)": round(df["elapsed"].mean(), 2),
    "95th percentile (ms)": round(df["elapsed"].quantile(0.95), 2),
    "Throughput (requests/sec)": round(len(df) / duration_seconds, 2)
}])
display(summary)

print("HTML report generated at /content/jmeter-report/index.html")
display(HTML("<p>Use the file browser in Colab to open or download the JMeter HTML report.</p>"))

## 6. Workshop discussion

Discuss:
- Why can an HTTP 200 response still require database-level validation?
- What do average response time and 95th-percentile response time tell us?
- What happens when a transfer is rejected?
- Why should a performance test use controlled, non-production test data?

**Important:** This lab is intentionally small. It demonstrates the testing workflow, not production-grade banking controls, distributed transaction processing, security testing, or realistic high-volume capacity.